# V14 Crisis Mining (Colab)

Crisis component of the V14 corpus (pairs with `v14_selfplay_colab.ipynb`; V13's recipe was majority-crisis: 5,526 crisis files vs 893 steady-state games). Each replay starts from a near-death snapshot of a policy-only probe and plays forward `--continue-turns 500` turns of strong-search MCTS — capturing both the recovery transition AND post-recovery steady-state play.

Player: **pillar3f + value_head_pillar3f + q_weight=2.0** — current best (HISTORY 167-168; 5k: mean 31.6k / P50 22.2k). The head is retrained on pillar3f's backbone per the HISTORY 158 rule — do NOT reuse older heads.

Settings (V14):
- recovery_turns 15 / prevention_turns 30 (snapshot offsets from death)
- recovery_sims 600 / prevention_sims 400 (same as V13's crisis@600/400)
- continue_turns 500 (strong-search post-snapshot play)
- **policy-probe cap 12000 turns** (pillar3f's median death ≈ 10.5k turns — probes must die to yield snapshots; expect ~half the probes to cap and yield nothing, so over-provision seeds)
- replay cap 25000, batch_size 8 (HISTORY 115)

**Upload to `MyDrive/alphatrain/` before running:**
1. `colorlines_pillar3d_v2.tar.gz` (current code tarball, already on Drive)
2. `pillar3f.pt` (~45 MB)
3. `value_head_pillar3f.pt` (~38 KB)

**Seed ranges:** M5 runs 1600000-1600999 locally; Colab instances take 1610000+ in 5K blocks (disjoint from V14 selfplay's 1500000+).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/alphatrain'

!cp {DRIVE}/colorlines_pillar3d_v2.tar.gz /content/
!cd /content && tar xzf colorlines_pillar3d_v2.tar.gz

os.makedirs('/content/alphatrain/data', exist_ok=True)
!cp {DRIVE}/pillar3f.pt /content/alphatrain/data/pillar3f.pt
!cp {DRIVE}/value_head_pillar3f.pt /content/alphatrain/data/value_head_pillar3f.pt
print(f'Model: {os.path.getsize("/content/alphatrain/data/pillar3f.pt")/1e6:.0f} MB')
print(f'Value head: {os.path.getsize("/content/alphatrain/data/value_head_pillar3f.pt")/1e3:.0f} KB')

!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

In [ ]:
# === V14 Crisis Mining ===
# M5 takes 1600000-1600999 locally. Colab instances split 1610000+ in 5K blocks:
#   Instance 1: 1610000-1615000
#   Instance 2: 1615000-1620000
SEED_START = 1610000
SEED_END = 1615000     # 5K probes per instance
# ============================

Q_WEIGHT = 2.0
RECOVERY_TURNS = 15
PREVENTION_TURNS = 30
RECOVERY_SIMS = 600
PREVENTION_SIMS = 400
CONTINUE_TURNS = 500    # post-snapshot play length — equilibrates back to steady-state
POLICY_MAX_TURNS = 12000  # pillar3f median death ~10.5k turns; probes must DIE to yield
MAX_TURNS = 25000
WORKERS = 20
BATCH_SIZE = 8          # HISTORY 115
SAVE_DIR = f'{DRIVE}/crisis_v14'

os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Seeds: {SEED_START}-{SEED_END-1} ({SEED_END-SEED_START} probes)')
print(f'q_weight: {Q_WEIGHT}, recovery {RECOVERY_TURNS}t @ {RECOVERY_SIMS} sims, prevention {PREVENTION_TURNS}t @ {PREVENTION_SIMS} sims')
print(f'Continue: {CONTINUE_TURNS} turns post-snapshot')
print(f'Policy probe cap: {POLICY_MAX_TURNS}, replay cap: {MAX_TURNS}, workers: {WORKERS}, bs: {BATCH_SIZE}')
print(f'Save: {SAVE_DIR}')

!python -m alphatrain.scripts.crisis_mining \
    --model /content/alphatrain/data/pillar3f.pt \
    --value-head-path /content/alphatrain/data/value_head_pillar3f.pt \
    --q-weight {Q_WEIGHT} \
    --seed-start {SEED_START} --seed-end {SEED_END} \
    --recovery-turns {RECOVERY_TURNS} --recovery-sims {RECOVERY_SIMS} \
    --prevention-turns {PREVENTION_TURNS} --prevention-sims {PREVENTION_SIMS} \
    --continue-turns {CONTINUE_TURNS} \
    --policy-max-turns {POLICY_MAX_TURNS} \
    --max-turns {MAX_TURNS} \
    --workers {WORKERS} --device cuda --batch-size {BATCH_SIZE} \
    --compile \
    --save-dir {SAVE_DIR}

## Notes

- **Two-phase pipeline:** Phase 1 = policy-only probes (now slower than V12-era — pillar3f probes run up to 12K turns). Phase 2 = MCTS replays (dominates wall time).
- **Replay yield:** pillar3f survives past the 12K-turn probe cap in roughly half the probes (median death ~10.5k turns) — those yield NO snapshots. Over-provision seed count accordingly; the output dir tells you the realized replay count.
- **Resume support:** re-running this cell skips already-completed `(seed, label)` pairs, so disconnects are safe.
- **Wall time:** longer than V12-era per seed (longer probes AND longer replays from healthier boards). Calibrate from the first ~100 seeds before sizing the campaign.
- **Watch:** the `[GPU] X evals, Y fwd (avg bs=Z, ZZZZ evals/s)` line. avg bs ≥50 and evals/s in low thousands means GPU is saturated; if either drops, check whether probes are all capping (Phase 2 starving for snapshots).
- **Output structure:** `game_seed{N}_recovery_score{S}.json`, `game_seed{N}_prevention_score{S}.json`. The V14 tensor build (M5) merges these with `selfplay_v14`: `build_expert_v2_tensor --games-dir data/selfplay_v14 data/crisis_v14 --policy-only-data --output alphatrain/data/v14_pillar3f.pt`.
- **M5 equivalent command** (seeds 1600000-1600999, runs in parallel with Colab shards):
  ```
  caffeinate -dim python -m alphatrain.scripts.crisis_mining \
      --model alphatrain/data/pillar3f.pt \
      --value-head-path alphatrain/data/value_head_pillar3f.pt \
      --q-weight 2.0 \
      --seed-start 1600000 --seed-end 1601000 \
      --recovery-turns 15 --recovery-sims 600 \
      --prevention-turns 30 --prevention-sims 400 \
      --continue-turns 500 \
      --policy-max-turns 12000 --max-turns 25000 \
      --workers 16 --device mps --batch-size 8 \
      --save-dir data/crisis_v14
  ```